# Exercise1

## ステップ1：似た機能のツールを2つ用意する

In [1]:
!python --version

Python 3.14.5


In [9]:
import json
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5"




In [5]:
tools_normal = [
    {
        "name": "search_open_tickets",
        "description": (
            "現在オープン（未解決）のサポートチケットを検索する。"
            "顧客が『今困っている問題』について尋ねたときに使う。"
            "解決済み・クローズ済みのチケットは対象外。"
        ),
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "search_ticket_history",
        "description": (
            "過去に解決・クローズされたチケットの履歴を検索する。"
            "『前回どう対応したか』『過去の事例』を尋ねられたときに使う。"
            "オープン中のチケットは対象外。"
        ),
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": "ポリシー上の判断が必要、または自動処理できない場合に人間へ引き継ぐ。",
        "input_schema": {
            "type": "object",
            "properties": {"reason": {"type": "string"}},
            "required": ["reason"],
        },
    },
]

tools_vague = [
    {
        "name": "search_open_tickets",
        "description": (
            "現在のチケットを検索する。"
        ),
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "search_ticket_history",
        "description": (
            "過去の履歴を検索する。"
        ),
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": "自動処理できない場合",
        "input_schema": {
            "type": "object",
            "properties": {"reason": {"type": "string"}},
            "required": ["reason"]
        },
    },
]

「今困っている」vs「過去にどう対応したか」を意図的に曖昧にした質問を3〜4個作り、モデルがどちらを選ぶか観察してください。description を弱くする（「チケットを検索する」とだけ書く）版と、上のように境界を明示した版で選択率が変わるはずです。今日の enum の実験と同じ構造です。

In [6]:
message_1 = "ログインできなくて困っています。"
message_2 = "先月も同じログインエラーが出て、その時どう直したか教えてください。"
message_3 = "また同じエラーが出ました。前回チケットを開いてもらった件です。"
message_4 = "この問題、そちらでは対応できないと思うので担当者に代わってください。"


In [ ]:
for name, msg in [("1",message_1),("2",message_2),("3",message_3),("4",message_4)]:
    for tools_name, tools in [("normal", tools_normal), ("vague", tools_vague)]:
        res = client.messages.create(
            model=model, max_tokens=500,
            messages=[{"role":"user","content":msg}],
            tools=tools,
        )
        call = next((b for b in res.content if b.type=="tool_use"), None)
        print(f"{name}/{tools_name}: {call.name if call else 'no tool call'}")

1/normal: search_open_tickets
1/vague: search_open_tickets
2/normal: search_ticket_history
2/vague: search_ticket_history
3/normal: search_ticket_history
3/vague: no tool call
4/normal: escalate_to_human
4/vague: escalate_to_human


In [11]:
for name, msg in [("1",message_1),("2",message_2),("3",message_3),("4",message_4)]:
    for tools_name, tools in [("normal", tools_normal), ("vague", tools_vague)]:
        res = client.messages.create(model=model, max_tokens=500,
    messages=[{"role":"user","content":message_3}], tools=tools_vague)
        for block in res.content:
            print(block.type, getattr(block, "text", None) or getattr(block, "name", None))
        print("stop_reason:", res.stop_reason)
        call = next((b for b in res.content if b.type=="tool_use"), None)
        print(f"{name}/{tools_name}: {call.name if call else 'no tool call'}")

text 前回開いたチケットの内容を確認するために、まず過去の履歴を検索させていただきます。
tool_use search_ticket_history
stop_reason: tool_use
1/normal: search_ticket_history
text 前回チケットを開いた件での同じエラーについてですね。状況をより詳しく把握するために、以下の情報を教えていただけますか？

1. **前回のチケット番号**（わかれば）
2. **エラーの内容や症状**（具体的な）
3. **いつ頃前回チケットを開いたのか**

または、前回のチケットについて検索するために、以下の情報のいずれかを教えてください：
- チケットに関連するキーワード
- エラーメッセージの一部
- 関連する製品やサービス名

これらの情報があれば、前回のチケット履歴を検索して、状況を確認し、適切に対応できます。
stop_reason: end_turn
1/vague: no tool call
text 申し訳ありませんが、前回のチケットについての詳細がわかりません。同じエラーの内容や、前回開いていただいたチケット番号などをご教示いただけますでしょうか？

それがあれば、以下の対応が可能です：

1. **エラーの具体的な内容** - どのようなエラーメッセージが表示されているのか
2. **前回のチケット番号** - または前回の対応時期
3. **どのような状況で発生しているのか** - 再現条件など

これらの情報をいただければ、チケット履歴を検索して、前回の対応内容を確認し、適切なサポートをさせていただきます。
stop_reason: end_turn
2/normal: no tool call
text 前回のチケットについて確認させていただきたいのですが、いくつか情報をお教えいただけますか？

1. **チケット番号**（わかればお願いします）
2. **エラーの内容**（どのようなエラーが出ているのか）
3. **関連するシステムやサービス名**（わかれば）

この情報があれば、前回のチケット履歴を検索して、現在のエラーとの関連性を調べることができます。

もし具体的な情報がなければ、エラーメッセージなど記憶に残っていることを教えていただければ、それで検索いたしま

temperature により、同じ入力・同じdescriptionでも「ツールを呼ぶ」「情報を聞き返す」の間で判断が揺れることがある。曖昧な依頼に対する挙動を安定させたい場合は、few-shotで模範例を1つ与える必要がある。どちらの挙動が「正解」かはユースケース次第。

## ステップ2：stop_reason で自分でループを回す

ここが今日の核心です。関数を実装し、tool_use の間は結果を積んで送り直し、end_turn で止める。

In [ ]:


def run_agent(user_message, tools, tool_impls, max_turns=5):
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        res = client.messages.create(
            model=model, max_tokens=1000,
            messages=messages, tools=tools,
        )
        messages.append({"role": "assistant", "content": res.content})

        if res.stop_reason == "end_turn":
            return res, turn

        if res.stop_reason == "tool_use":
            tool_results = []
            for block in res.content:
                if block.type != "tool_use":
                    continue
                impl = tool_impls[block.name]
                result = impl(**block.input)   # ここが構造化エラー設計の入口
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })
            messages.append({"role": "user", "content": tool_results})
            continue

        # tool_use でも end_turn でもない場合（max_tokens 等）
        return res, turn

    raise RuntimeError("max_turns に達しました")

In [12]:
def search_open_tickets(query):
    return {
        "results": [
            {"ticket_id": "OPEN-1042", "summary": f"{query} に関する未解決の問い合わせ", "status": "open"}
        ]
    }

def search_ticket_history(query):
    return {
        "results": [
            {"ticket_id": "HIST-0817", "summary": f"{query} は前回、設定リセットで解決", "status": "closed"}
        ]
    }

def escalate_to_human(reason):
    return {"escalated": True, "reason": reason, "assigned_to": "human-queue"}

tool_impls = {
    "search_open_tickets": search_open_tickets,
    "search_ticket_history": search_ticket_history,
    "escalate_to_human": escalate_to_human,
}

In [13]:
for name, msg in [("1",message_1),("2",message_2),("3",message_3),("4",message_4)]:
    for tools_name, tools in [("normal", tools_normal), ("vague", tools_vague)]:
        res, turns = run_agent(msg, tools, tool_impls)
        for block in res.content:
            print(block.type, getattr(block, "text", None) or getattr(block, "name", None))
        print(f"{name}/{tools_name} stop_reason:", res.stop_reason, "turns:", turns)

text ログインできないというお問題について確認しました。

**現在の状況：**
- 同様のお悩みで現在対応中のチケット（OPEN-1042）があります
- 過去には「設定リセット」で解決した事例（HIST-0817）があります

**一般的な解決方法としては、以下をお試しいただけます：**

1. **パスワードのリセット** - 「パスワードを忘れた」機能を使用
2. **ブラウザのキャッシュクリア** - クッキーやキャッシュをクリアしてから再度ログイン
3. **別のブラウザで試す** - ブラウザ固有の問題でないか確認
4. **設定リセット** - 過去の事例で有効だった方法です

それでも解決しない場合は、詳しい状況（エラーメッセージの内容、使用端末、ブラウザなど）を教えていただければ、より詳細なサポートができます。

いかがでしょうか？
1/normal stop_reason: end_turn turns: 1
text 検索結果から、ログインできない問題についての過去の事例が見つかりました。

**確認した情報:**
- **現在の関連チケット**: OPEN-1042（未解決）
- **過去の解決事例**: HIST-0817（設定リセットで解決）

ログインできない問題について、以下の一般的な対処方法をお試しください：

1. **パスワードの確認**
   - キャップスロックが有効になっていないか確認
   - パスワードを正確に入力しているか再確認

2. **ブラウザのキャッシュをクリア**
   - ブラウザのキャッシュとクッキーを削除してから再度ログイン

3. **別のブラウザで試す**
   - 別のブラウザやシークレットモードで試す

4. **パスワードのリセット**
   - 過去の解決事例では設定リセットで解決しているため、パスワードリセット機能をお試しください

これらの方法で解決しない場合、またはご質問があれば、人間のサポート担当者に引き継ぎさせていただきます。いかがでしょうか？
1/vague stop_reason: end_turn turns: 1
text ご質問ありがとうございます。

過去のチケット履歴を確認しました。先月のログインエラーについては、**設定リセット**で解決していたようです。（

In [14]:
def run_agent(user_message, tools, tool_impls, max_turns=5):
    messages = [{"role": "user", "content": user_message}]
    for turn in range(max_turns):
        res = client.messages.create(model=model, max_tokens=1000, messages=messages, tools=tools)
        messages.append({"role": "assistant", "content": res.content})
        if res.stop_reason == "end_turn":
            return res, turn, messages
        if res.stop_reason == "tool_use":
            tool_results = []
            for block in res.content:
                if block.type != "tool_use":
                    continue
                print("  [tool called]", block.name, block.input)   # ← 追加
                result = tool_impls[block.name](**block.input)
                tool_results.append({"type": "tool_result", "tool_use_id": block.id,
                                      "content": json.dumps(result, ensure_ascii=False)})
            messages.append({"role": "user", "content": tool_results})
            continue
        return res, turn, messages
    raise RuntimeError("max_turns に達しました")

In [16]:
for name, msg in [("1",message_1),("2",message_2),("3",message_3),("4",message_4)]:
    for tools_name, tools in [("normal", tools_normal), ("vague", tools_vague)]:
        print(f"--- {name}/{tools_name} ---")
        res, turns, messages = run_agent(msg, tools, tool_impls)
        for block in res.content:
            print(block.type, getattr(block, "text", None) or getattr(block, "name", None))
        print(f"{name}/{tools_name} stop_reason:", res.stop_reason, "turns:", turns)

--- 1/normal ---
  [tool called] search_open_tickets {'query': 'ログインできない'}
text ログインできない問題についてはいくつかのケースがあります。以下の点をご確認ください：

**よくある原因と対応方法：**

1. **パスワードの確認**
   - Caps Lock（大文字ロック）がONになっていないか
   - パスワードが正しく入力されているか

2. **アカウント関連**
   - メールアドレスが正しいか
   - アカウントがロックされていないか

3. **ブラウザ・キャッシュ**
   - ブラウザのキャッシュをクリアしてお試しください
   - 別のブラウザでのお試し

4. **パスワードリセット**
   - ログイン画面の「パスワードを忘れた」からリセットできます

上記の方法で解決しない場合は、より詳しい状況をお知らせいただくことで、サポートをさせていただけます。

- **具体的なエラーメッセージ**は表示されていますか？
- **いつ頃からログインできなくなった**のでしょうか？
- **使用しているデバイス・ブラウザ**は何ですか？

これらの情報をいただければ、より適切なサポートが可能です。
1/normal stop_reason: end_turn turns: 1
--- 1/vague ---
  [tool called] search_open_tickets {'query': 'ログイン できない'}
  [tool called] search_ticket_history {'query': 'ログイン できない'}
text 確認した結果をお知らせします：

**現在のチケット：**
- チケットID: OPEN-1042
- 状況：ログインできない問題が現在未解決のまま報告されています

**過去の解決事例：**
- チケットID: HIST-0817
- 前回同様の問題は「設定リセット」で解決されています

ログインできない問題については、以下のような一般的な対応方法があります：

1. **パスワードリセット** - パスワードを忘れた場合
2. **設定リセット** - 過去の解決事例と同じ方法
3. **ブラウザのキャッシュ

・stop_reasonでのループ配線は正常動作（②の目的達成）
・vague toolsでは並列tool_use（安全側に倒れ両方呼ぶ）が起きうる
・同一プロンプト・同一toolsでも複数回実行すると
  「ツール呼び出しあり/なし」が割れるケースがある(message_3)
・エスカレーション後もモデルが対応を続けようとする一貫性の欠如が見られた

## 3. 構造化エラー設計

In [29]:
FORCE_TRANSIENT = {"n": 0}  # 最初の呼び出しだけ強制的にtransientにする


def search_open_tickets(query):
    if not query.strip():
        return {"error": {"category": "validation", "isRetryable": True,
                           "message": "queryが空です"}}
    # 例: 一時的な障害を模擬
    import random
    if random.random() < 0.3:
        return {"error": {"category": "transient", "isRetryable": True,
                           "message": "検索インデックスに一時的に接続できません"}}
    return {
        "results": [
            {"ticket_id": "OPEN-1042", "summary": f"{query} に関する未解決の問い合わせ", "status": "open"}
        ]
    }

def search_ticket_history(query):
    if not query.strip():
        return {"error": {"category": "validation", "isRetryable": False,
                           "message": "queryが空です"}}
    if FORCE_TRANSIENT["n"] < 1:
        FORCE_TRANSIENT["n"] += 1
        return {"error": {"category": "transient", "isRetryable": True,
                           "message": "検索インデックスに一時的に接続できません"}}

    return {
        "results": [
            {"ticket_id": "HIST-0817", "summary": f"{query} は前回、設定リセットで解決", "status": "closed"}
        ]
    }

def escalate_to_human(reason):
    if not reason.strip():
        return {"error": {"category": "validation", "isRetryable": False,
                           "message": "reasonが空です。理由を明記してください"}}
    if "課金" in reason or "billing" in reason.lower():
        return {"error": {"category": "permission", "isRetryable": False,
                           "message": "課金関連の問題は billing チームへの直接エスカレーションが必要です"}}
    import random
    if random.random() < 0.3:
        return {"error": {"category": "transient", "isRetryable": True,
                           "message": "エスカレーションキューに一時的に接続できません"}}
    return {"escalated": True, "reason": reason, "assigned_to": "human-queue"}

tool_impls = {
    "search_open_tickets": search_open_tickets,
    "search_ticket_history": search_ticket_history,
    "escalate_to_human": escalate_to_human,
}

In [24]:
def run_agent(user_message, tools, tool_impls, max_turns=5):
    messages = [{"role": "user", "content": user_message}]
    for turn in range(max_turns):
        res = client.messages.create(model=model, max_tokens=1000, messages=messages, tools=tools)
        messages.append({"role": "assistant", "content": res.content})
        if res.stop_reason == "end_turn":
            return res, turn, messages
        if res.stop_reason == "tool_use":
            tool_results = []
            for block in res.content:
                if block.type != "tool_use":
                    continue
                result = tool_impls[block.name](**block.input)
                is_err = "error" in result
                print("  [tool called]", block.name, block.input,
                      "-> ERROR:" + result["error"]["category"] if is_err else "-> OK")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result, ensure_ascii=False),
                    "is_error": is_err,   # ← Anthropic API のtool_resultにもこのフィールドがある
                })
            messages.append({"role": "user", "content": tool_results})
            continue
        return res, turn, messages
    raise RuntimeError("max_turns に達しました")

In [25]:
for name, msg in [("1",message_1),("2",message_2),("3",message_3),("4",message_4)]:
    for tools_name, tools in [("normal", tools_normal), ("vague", tools_vague)]:
        print(f"--- {name}/{tools_name} ---")
        res, turns, messages = run_agent(msg, tools, tool_impls)
        for block in res.content:
            print(block.type, getattr(block, "text", None) or getattr(block, "name", None))
        print(f"{name}/{tools_name} stop_reason:", res.stop_reason, "turns:", turns)

--- 1/normal ---
  [tool called] search_open_tickets {'query': 'ログイン'} -> OK
  [tool called] search_ticket_history {'query': 'ログインできない'} -> ERROR:transient
text ログイン問題についてのオープンチケットが存在しているようです。この問題はお客様にとって重要なものですので、詳しく確認させていただきたいのですが、以下の情報を教えていただけますか？

1. **エラーメッセージ** - ログイン時にどのようなエラーメッセージが表示されていますか？
2. **アカウント情報** - どのサービスへのログインですか？（例：Webサイト、アプリなど）
3. **最後にログインできた時期** - いつまでログインできていましたか？
4. **試みた対応** - パスワードリセットなど何か試しましたか？

詳細をいただければ、より適切なサポートができます。複雑な対応が必要な場合は、専門チームに引き継ぐこともできます。
1/normal stop_reason: end_turn turns: 1
--- 1/vague ---
  [tool called] search_open_tickets {'query': 'ログインできない'} -> ERROR:transient
  [tool called] search_ticket_history {'query': 'ログインできない'} -> OK
text 過去の履歴から、ログインできないという問題は**設定リセット**で解決したケースがあります。

お困りの状況をより詳しく教えていただけますか？以下の点を教えていただけると、より的確なサポートができます：

1. **どのサービス/アプリケーション**にログインできないのか
2. **エラーメッセージ**が表示されているか（表示されている場合はどのようなメッセージか）
3. **どのような操作**をしたときにログインできなくなったのか
4. **ユーザーID/メールアドレス**は確実に正しく入力されているか
5. **Caps Lock**や**言語設定**に問題がないか

これらの情報をいただければ、より迅速に問題

In [ ]:
FORCE_TRANSIENT = {"n": 0}  # 最初の呼び出しだけ強制的にtransientにする

def run_agent(user_message, tools, tool_impls, max_turns=5):
    messages = [{"role": "user", "content": user_message}]
    for turn in range(max_turns):
        res = client.messages.create(model=model, max_tokens=1000, messages=messages, tools=tools)
        messages.append({"role": "assistant", "content": res.content})
        if res.stop_reason == "end_turn":
            return res, turn, messages
        if res.stop_reason == "tool_use":
            tool_results = []
            for block in res.content:
                if block.type != "tool_use":
                    continue
                result = tool_impls[block.name](**block.input)
                if ("error" in result
                        and result["error"]["category"] == "transient"
                        and result["error"]["isRetryable"]):
                    print("    [app-level retry]", block.name)
                    result = tool_impls[block.name](**block.input)
                is_err = "error" in result
                print("  [tool called]", block.name, block.input,
                      "-> " + (f"ERROR:{result['error']['category']}" if is_err else "OK"))
                tool_results.append({
                    "type": "tool_result", "tool_use_id": block.id,
                    "content": json.dumps(result, ensure_ascii=False), "is_error": is_err,
                })
            messages.append({"role": "user", "content": tool_results})
            continue
        return res, turn, messages
    raise RuntimeError("max_turns に達しました")

for name, msg in [("1",message_1),("2",message_2),("3",message_3),("4",message_4)]:
    for tools_name, tools in [("normal", tools_normal), ("vague", tools_vague)]:
        print(f"--- {name}/{tools_name} ---")
        res, turns, messages = run_agent(msg, tools, tool_impls)
        print(f"{name}/{tools_name} stop_reason:", res.stop_reason, "turns:", turns)

--- 1/normal ---
    [app-level retry] search_open_tickets
  [tool called] search_open_tickets {'query': 'ログインできない'} -> OK
    [app-level retry] search_ticket_history
  [tool called] search_ticket_history {'query': 'ログインできない'} -> OK
1/normal stop_reason: end_turn turns: 1
--- 1/vague ---
  [tool called] search_open_tickets {'query': 'ログインできない'} -> OK
  [tool called] search_ticket_history {'query': 'ログインできない'} -> OK
1/vague stop_reason: end_turn turns: 1
--- 2/normal ---
  [tool called] search_ticket_history {'query': 'ログインエラー'} -> OK
2/normal stop_reason: end_turn turns: 1
--- 2/vague ---
  [tool called] search_ticket_history {'query': 'ログインエラー'} -> OK
2/vague stop_reason: end_turn turns: 1
--- 3/normal ---
3/normal stop_reason: end_turn turns: 0
--- 3/vague ---
  [tool called] search_ticket_history {'query': 'エラー'} -> OK
3/vague stop_reason: end_turn turns: 1
--- 4/normal ---
    [app-level retry] escalate_to_human
  [tool called] escalate_to_human {'reason': '顧客からの依頼により、人間の担当者への引き継ぎが

Exercise 1:
・ツール選択はdescriptionの強弱だけでなく、同一条件でも試行毎に揺れることがある
  （特に境界的な質問文で顕著。1回の試行で判断しない）
・vague toolsでは並列tool_use（安全側の判断）が誘発されることがある
・transientエラーはモデルへの提示ではなく、アプリ層での自動吸収が正しい設計
・エスカレーション後もモデルが追加ヒアリングを続けるなど、
  役割の一貫性が崩れる場合がある

TOOLの選択はDescription記載内容によって左右され、VagueなDescriptionでは呼び出されないこともあった(確率的)。また、Errorが出た際に、Retriableなどと伝えてもRetlyすることを保証しない。サービスの提供という視点では、エラー時にはApp側で「retryableなものは自動リトライで吸収し、それでも解決しないものやvalidation/permissionは人間対応へ回す」というエラー種別に応じた分岐が品質向上に役立つ。

## ステップ4：programmatic hook

In [31]:
def process_refund(order_id, amount):
    return {"refunded": True, "order_id": order_id, "amount": amount}

tool_impls["process_refund"] = process_refund

tools_with_refund = tools_normal + [{
    "name": "process_refund",
    "description": "注文の返金処理を行う。金額が大きい場合は人間の承認が必要になることがある。",
    "input_schema": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string"},
            "amount": {"type": "number"}
        },
        "required": ["order_id", "amount"]
    }
}]

def enforce_business_rules(block):
    """ツール実行前のフック。ビジネスルール違反を機械的にブロックする"""
    if block.name == "process_refund" and block.input.get("amount", 0) > 50000:
        return {"error": {"category": "permission", "isRetryable": False,
                           "message": "5万円を超える返金は自動処理できません。escalate_to_humanを使って人間の承認を得てください。"}}
    return None

In [32]:
def run_agent(user_message, tools, tool_impls, max_turns=5):
    messages = [{"role": "user", "content": user_message}]
    for turn in range(max_turns):
        res = client.messages.create(model=model, max_tokens=1000, messages=messages, tools=tools)
        messages.append({"role": "assistant", "content": res.content})
        if res.stop_reason == "end_turn":
            return res, turn, messages
        if res.stop_reason == "tool_use":
            tool_results = []
            for block in res.content:
                if block.type != "tool_use":
                    continue
                blocked = enforce_business_rules(block)          # ← フック
                if blocked:
                    result = blocked
                    print("  [hook blocked]", block.name, block.input)
                else:
                    result = tool_impls[block.name](**block.input)
                    if ("error" in result and result["error"]["category"] == "transient"
                            and result["error"]["isRetryable"]):
                        result = tool_impls[block.name](**block.input)
                is_err = "error" in result
                print("  [tool called]", block.name, block.input,
                      "-> " + (f"ERROR:{result['error']['category']}" if is_err else "OK"))
                tool_results.append({
                    "type": "tool_result", "tool_use_id": block.id,
                    "content": json.dumps(result, ensure_ascii=False), "is_error": is_err,
                })
            messages.append({"role": "user", "content": tool_results})
            continue
        return res, turn, messages
    raise RuntimeError("max_turns に達しました")

In [33]:
res, turns, messages = run_agent(
    "注文ID ORD-5521 の返金を8万円でお願いします。",
    tools_with_refund, tool_impls
)
for block in res.content:
    print(block.type, getattr(block, "text", None))

  [hook blocked] process_refund {'order_id': 'ORD-5521', 'amount': 80000}
  [tool called] process_refund {'order_id': 'ORD-5521', 'amount': 80000} -> ERROR:permission
  [tool called] escalate_to_human {'reason': '注文ID ORD-5521 の80,000円の返金申請。ポリシーにより5万円を超える返金は人間の承認が必要。'} -> OK
text ご対応ありがとうございます。注文ID **ORD-5521** の **80,000円の返金申請**は、金額が大きいため人間のオペレーターによる審査が必要です。

申請内容は人間のキューに引き継がれました。専門のオペレーターが確認の上、対応させていただきます。

何かご不明な点がございましたら、お気軽にお問い合わせください。


まとめ<br>
block.name でどのツールかを判定し、block.input（JSON Schemaで定義された構造）から値を読み、ツール本体を実行する前にブロックするかを機械的に決定する。これが Programmatic Hook（PreToolUse相当）。

## ステップ5：multi-concern message

In [34]:
message_multi = "ログインできない問題と、先月の請求金額が間違っている件、両方対応してほしいです。"

res, turns, messages = run_agent(message_multi, tools_normal, tool_impls)
for block in res.content:
    print(block.type, getattr(block, "text", None) or getattr(block, "name", None))
print("stop_reason:", res.stop_reason, "turns:", turns)

  [tool called] search_open_tickets {'query': 'ログインできない'} -> OK
  [tool called] search_open_tickets {'query': '請求金額 間違い'} -> OK
text 確認いただきました。現在、以下の2つのご問題について対応状況をお知らせします：

**対応中のチケット：**
- **OPEN-1042**: ログインできない問題について（未解決）
- **OPEN-1042**: 請求金額の間違いについて（未解決）

両方のご問題が現在オープンな状態で対応待ちとなっています。

**次のステップとしては：**

1. **ログインできない問題** - アカウント設定やパスワードリセット、セッション情報などを確認する必要があります
2. **請求金額の誤りについて** - 先月の明細確認や料金体系の見直しが必要です

これらの問題は複数の対応項目を含むため、サポートチームの詳細確認が必要です。具体的な以下の情報をいただけると、より迅速に対応できます：

- **ログイン問題**: どのような画面でエラーが出ているか、エラーメッセージは何か
- **請求金額**: どの項目がいくら間違っていると考えられるか、期待していた金額は何か

詳しい内容をお聞かせいただければ、より正確なサポートをさせていただきます。
stop_reason: end_turn turns: 1
